# DeepScaler: train, so sánh baseline và tune tham số (dữ liệu có burst)

Tái lập các bước đã chạy ngày 04/11/2026; kết quả nằm trong `experiments/RESULTS_burst_20261104.md` và `experiments/RESULTS_free_tune_20261104.md`.

1. **Fig. 12, giữ giá trị của code:** DeepScaler (5 epoch × 5 vòng, cửa sổ 80) so với DCRNN, T-GCN và cách giữ nguyên số pod, lấy mẫu 30 s và 60 s, 3 seed (`experiments/tune_grid.py`).
2. **Tune tự do:** được đổi giá trị của code vì cluster khác báo (Sec. V-A). Chọn cấu hình theo MAE trên **tập validation**, rồi chạy 3 seed (`experiments/tune_free.py`).
3. **Baseline cùng cửa sổ 20** với cấu hình tốt nhất, để so công bằng.
4. **Đóng gói và tải về** model đã tune, dùng được ngay cho `guide/RUN_ONLINE.md`.

**Trước khi chạy:**
- Chọn **Thời gian chạy → Thay đổi loại thời gian chạy → GPU T4**.
- Chuẩn bị 3 file do `pipeline/collect.py` tạo ra: `<run>.csv`, `<run>_od.npy`, `<run>_npas.npy` (thư mục `data/raw/` trên máy bạn).

**Thời gian trên T4:**
- Bước 1: khoảng 25 phút.
- Bước 2: khoảng 70 phút.
- Bước 3: khoảng 10 phút.

Mọi thứ trong `/content` mất khi phiên Colab kết thúc, nên phải tải kết quả về (bước 4, 5) trước khi đóng.

In [ ]:
# ==== Cấu hình ====
RUN    = "train_burst_20261104_0751"   # tên lần thu (không có đuôi)
BRANCH = "deepscaler-pipeline"
SEEDS  = "2021,2022,2023"

## 0. Lấy code và kiểm tra GPU

In [ ]:
import os, subprocess
if not os.path.exists("/content/DeepScaler"):
    !git clone -q -b {BRANCH} https://github.com/maintn24/DeepScaler.git /content/DeepScaler
%cd /content/DeepScaler
!git pull -q && git log --oneline -1
!nvidia-smi -L
!pip -q install networkx pyyaml

## 1. Upload dữ liệu

Chạy ô dưới rồi chọn 3 file `<RUN>.csv`, `<RUN>_od.npy`, `<RUN>_npas.npy`.

Cách khác: kéo thả các file vào thanh **Tệp** bên trái. File sẽ nằm ở `/content`.

In [ ]:
from google.colab import files
need = [f"{RUN}.csv", f"{RUN}_od.npy", f"{RUN}_npas.npy"]
if not all(os.path.exists(f"/content/{f}") for f in need):
    up = files.upload()
    for name, data in up.items():
        open(f"/content/{name}", "wb").write(data)
for f in need:
    print(f, "OK" if os.path.exists(f"/content/{f}") else "THIẾU")
RAW, OD, NPAS = (f"/content/{f}" for f in need)

## 2. (Tùy chọn) Kiểm tra chất lượng dữ liệu

In tỉ lệ vượt SLA, độ biến thiên số pod và MAE của cách giữ nguyên số pod. Phần nhãn nhu cầu chỉ hiện nếu có `reports/mu.json`.

In [ ]:
!python -m pipeline.check_data {RAW} --slo 0.2

## 3. Fig. 12 với giá trị của code

DeepScaler 5/5 (`predict_scale.py`), lấy mẫu 30 s và 60 s, chia tập 0,7/0,1/0,2. DCRNN và T-GCN được train tối đa 300 epoch, dừng sớm sau 20 epoch không cải thiện.

Lệnh chạy nền và ghi log ra `/content/grid.log`. Chạy ô "theo dõi" để xem tiến độ.

In [ ]:
P_grid = subprocess.Popen(
    f'python experiments/tune_grid.py --raw {RAW} --od {OD} --npas {NPAS} --out /content/grid '
    f'--strides 1,2 --splits "0.7,0.1,0.2" --epoch_iters 5/5 --seeds {SEEDS} > /content/grid.log 2>&1', shell=True)

In [ ]:
# theo dõi: chạy lại ô này bao nhiêu lần cũng được
!tail -4 /content/grid.log; ls /content/grid/DONE 2>/dev/null && echo "XONG"

## 4. Tune tự do (chọn theo validation) — chạy sau khi bước 3 XONG để không tranh GPU

In [ ]:
P_free = subprocess.Popen(
    f'python experiments/tune_free.py --raw {RAW} --od {OD} --npas {NPAS} --out /content/free '
    f'--seeds {SEEDS} > /content/free.log 2>&1', shell=True)

In [ ]:
# theo dõi
!tail -6 /content/free.log; ls /content/free/DONE 2>/dev/null && echo "XONG"

## 5. Baseline với cửa sổ của cấu hình tốt nhất (so công bằng) — sau khi bước 4 XONG

In [ ]:
import json
best = json.load(open("/content/free/best.json"))
DATA_BEST = f"/content/free/data_in{best['step_in']}"
print("best:", best)
P_bl = subprocess.Popen(
    f'for m in DCRNN TGCN; do for s in {SEEDS.replace(",", " ")}; do '
    f'python train_baselines.py --data_dir {DATA_BEST} --model $m --seed $s '
    f'--out_dir /content/free/blbest_${{m}}_s$s --max_epoch 300 --patience 20 '
    f'> /content/free/blbest_${{m}}_s$s.log 2>&1; echo $m $s >> /content/blbest.log; done; done',
    shell=True, executable="/bin/bash")

In [ ]:
!cat /content/blbest.log 2>/dev/null

## 6. Bảng tổng hợp

In [ ]:
import glob, numpy as np, pandas as pd
pd.set_option("display.width", 250)

def load(pattern):
    rows = []
    for f in sorted(glob.glob(pattern)):
        m = json.load(open(f)); run = f.split("/")[-2]
        for model, r in m["results"]["test"].items():
            rows.append({"run": run, "model": model, "mae": r["mae"]})
    return pd.DataFrame(rows)

def group(run):          # bỏ hậu tố _s<seed>
    return run.rsplit("_s", 1)[0]

d = pd.concat([load("/content/grid/*/metrics.json"), load("/content/free/final_*/metrics.json"),
               load("/content/free/blbest_*/metrics.json")], ignore_index=True)
d["config"] = d["run"].map(group)
t = d.groupby(["config", "model"])["mae"].agg(["mean", "std", "count"]).round(3)
print(t.to_string())

In [ ]:
# bước 1 của tune tự do (seed 2021): MAE valid/test từng yếu tố
r = pd.DataFrame(json.load(open("/content/free/results.json")))
print(r[["name", "seed", "valid_mae", "test_mae", "test_persist", "edges", "sim_od", "train_s"]].round(3).to_string())

## 7. Đóng gói model tốt nhất và tải về

Chọn seed có **MAE validation thấp nhất** trong 3 lần chạy `final_best`; không chọn theo tập test.

File `ds_burst_tuned.zip` chứa:
- `model_states/ds_burst_tuned/` (model, cấu hình, scaler, metadata);
- `data/burst_in<step_in>/`.

Trên máy bạn, chạy `unzip -o ds_burst_tuned.zip` trong repo. Khi chạy online: `MODEL_DIR=model_states/ds_burst_tuned deploy/04_run_deepscaler.sh`.

In [ ]:
fin = [x for x in json.load(open("/content/free/results.json")) if x["name"] == "final_best"]
pick = min(fin, key=lambda x: x["valid_mae"]); s = pick["seed"]
print("chọn seed", s, "valid", round(pick["valid_mae"], 3), "test", round(pick["test_mae"], 3))
!rm -rf /content/pkg && mkdir -p /content/pkg/model_states /content/pkg/data
!cp -r /content/free/final_best_s{s} /content/pkg/model_states/ds_burst_tuned
!cp /content/free/final_best_s{s}_model.yaml /content/free/final_best_s{s}_train.yaml /content/free/best.json /content/free/results.json /content/pkg/model_states/ds_burst_tuned/
!cp -r {DATA_BEST} /content/pkg/data/burst_in{best['step_in']}
!cd /content/pkg && rm -f /content/ds_burst_tuned.zip && zip -qr /content/ds_burst_tuned.zip . && ls -lh /content/ds_burst_tuned.zip
files.download("/content/ds_burst_tuned.zip")

In [ ]:
# (tùy chọn) tải toàn bộ kết quả: log, metrics.json của mọi lần chạy (không kèm model)
!cd /content && zip -qr results_all.zip grid free -i '*.json' '*.log' && ls -lh results_all.zip
files.download("/content/results_all.zip")

## 8. Nhãn nhu cầu (O1): build dataset, train `ds_demand`, kiểm tra ACI offline

Mục này dùng cho phần của luận văn và **chạy độc lập với các bước 2–7**: chỉ cần chạy ô Cấu hình, bước 0 và bước 1 (upload `<RUN>.csv`, `_od.npy`, `_npas.npy`) trước.

- **Nhãn:** feature 0 là `demand = max(1, ceil(req/μ))`, với μ đo theo SLA (`reports/mu_sla.json`, do `deploy/07_profile_sla.sh` tạo trên EC2), thay cho số pod mà HPA chọn lúc thu dữ liệu (nhãn của báo, Sec. III-A).
- **Tham số model:** cấu hình đã tune trên dữ liệu burst (`experiments/RESULTS_free_tune_20261104.md`): cửa sổ 20, 3 block, 128 filter, kernel 5, batch 32, lr 1e-3, 50 epoch × 6 vòng, ε 0,1, δ 0,2. Nếu bước 4 đã chạy trong phiên này, cấu hình lấy từ `/content/free/best.json`.
- **Kết quả:** `model_states/ds_demand`, dùng cho `deploy/05_online_run.sh aci` (`ACI_MODEL_DIR=model_states/ds_demand`).

In [ ]:
# ==== Cấu hình mục 8 ====
MU_FILE      = "reports/mu_sla.json"     # upload từ máy bạn (reports/mu_sla.json)
SEEDS_DEMAND = "2021,2022,2023"           # chọn seed có MAE validation thấp nhất
ALPHA        = 0.05                       # mục tiêu tỉ lệ thiếu cho aci_replay

import os, json
os.chdir("/content/DeepScaler")
if not os.path.exists(MU_FILE):
    from google.colab import files
    up = files.upload()                   # chọn mu_sla.json
    os.makedirs("reports", exist_ok=True)
    for name, data in up.items():
        open(MU_FILE, "wb").write(data)
mu = json.load(open(MU_FILE))
print({s[:12]: (round(v["mu"], 1), v.get("method")) for s, v in mu.items()})

In [ ]:
# build dataset: feature 0 = demand (thesis O1); cùng cách chia và cửa sổ với model đã tune
!python -m pipeline.build_dataset --raw {RAW} --od_adj {OD} --npas {NPAS} \
    --step_in 20 --split 0.7,0.1,0.2 --split_by target \
    --features demand,cpu,res,req,mem --mu {MU_FILE} --out_dir /content/data_demand
import numpy as np
m = json.load(open("/content/data_demand/meta.json"))
print("features:", m["features"], "| step_in:", m["step_in"], "| samples:", m["samples"])
z = np.load("/content/data_demand/train.npz"); print("arrays:", z.files)
y = z[z.files[-1]] if "y" not in z.files else z["y"]
print("demand label: min", float(y.min()), "max", float(y.max()))

In [ ]:
# cấu hình model/train đã tune (lấy từ bước 4 nếu có, nếu không dùng giá trị trong RESULTS_free_tune_20261104.md)
import yaml, copy
best = json.load(open("/content/free/best.json")) if os.path.exists("/content/free/best.json") else \
       dict(epoch=50, iter=6, step_in=20, lr=1e-3, filters=128, blocks=3, kernel=5, batch=32, adj="od", epsilon=0.1, delta=0.2)
print("config:", best)
assert int(best["step_in"]) == 20, "best.json có step_in khác 20: build lại dataset với --step_in tương ứng"
M0, T0 = yaml.safe_load(open("config/train_dataset_speed.yaml")), yaml.safe_load(open("config/train_config.yaml"))
def write_cfg(seed):
    mm, tt = copy.deepcopy(M0), copy.deepcopy(T0)
    mm["AdapGLA"].update(num_cheb_filter=int(best["filters"]), num_time_filter=int(best["filters"]),
                         num_block=int(best["blocks"]), kernel_size=int(best["kernel"]))
    mm["dataset"]["batch_size"] = int(best["batch"])
    tt.update(seed=seed, learning_rate=float(best["lr"])); tt["lr_scheduler"]["T_max"] = int(best["epoch"])
    yaml.safe_dump(mm, open(f"/content/dm_model_{seed}.yaml", "w")); yaml.safe_dump(tt, open(f"/content/dm_train_{seed}.yaml", "w"))

In [ ]:
# train: mỗi seed khoảng 3 phút trên T4
import subprocess
res = {}
for seed in [int(s) for s in SEEDS_DEMAND.split(",")]:
    write_cfg(seed)
    out = f"/content/ds_demand_s{seed}"
    cmd = ["python", "train.py", "--data_dir", "/content/data_demand", "--out_dir", out,
           "--model_config_path", f"/content/dm_model_{seed}.yaml", "--train_config_path", f"/content/dm_train_{seed}.yaml",
           "--num_epoch", str(best["epoch"]), "--num_iter", str(best["iter"]), "--adj", str(best["adj"])]
    for k in ("epsilon", "delta"):
        if best.get(k) is not None: cmd += [f"--{k}", str(best[k])]
    r = subprocess.run(cmd, capture_output=True, text=True); open(out + ".log", "w").write(r.stdout + r.stderr)
    if r.returncode: print("seed", seed, "LỖI, xem", out + ".log"); continue
    mt = json.load(open(f"{out}/metrics.json"))["results"]
    res[seed] = {sp: {k: round(v["mae"], 3) for k, v in mt[sp].items()} for sp in ("valid", "test")}
    print(seed, res[seed])

In [ ]:
# chọn seed theo MAE validation (không theo test), chép thành model_states/ds_demand
import shutil, numpy as np
pick = min(res, key=lambda s: res[s]["valid"]["DeepScaler(AdapGLA)"])
for sp in ("valid", "test"):
    ds = [res[s][sp]["DeepScaler(AdapGLA)"] for s in res]; ps = [res[s][sp]["Persistence(last pods)"] for s in res]
    print(f"{sp}: DeepScaler MAE {np.mean(ds):.3f} ± {np.std(ds):.3f} | persistence {np.mean(ps):.3f}")
print("chọn seed", pick, res[pick])
shutil.rmtree("model_states/ds_demand", ignore_errors=True)
shutil.copytree(f"/content/ds_demand_s{pick}", "model_states/ds_demand")
json.dump({"seed": pick, "config": best, "results": res}, open("model_states/ds_demand/selection.json", "w"), indent=1)

In [ ]:
# kiểm tra offline: luật làm tròn của DeepScaler vs ACI toàn ứng dụng, trên cùng dự báo
!python -m pipeline.aci_replay --csv {RAW} --mu {MU_FILE} --forecaster model --model_dir model_states/ds_demand --alpha {ALPHA}
!python -m pipeline.aci_replay --csv {RAW} --mu {MU_FILE} --forecaster persistence --alpha {ALPHA}

Đọc kết quả `aci_replay`:
- Cột `miss_any` là tỉ lệ bước có ít nhất một dịch vụ trong V_act thiếu pod so với nhu cầu. Dòng `aci` phải gần `ALPHA`; dòng `deepscaler` (luật làm tròn gốc) thường vượt xa.
- Cột `replicas` là số pod trung bình, so với cột `demand`.
- Chạy với `persistence` để kiểm tra H4: lớp ACI vẫn đạt mục tiêu khi thay model bằng một dự báo đơn giản.

In [ ]:
# đóng gói và tải về: giải nén trong repo (unzip -o ds_demand.zip), rồi gửi lên EC2
!cd /content/DeepScaler && rm -f /content/ds_demand.zip && zip -qr /content/ds_demand.zip model_states/ds_demand && ls -lh /content/ds_demand.zip
from google.colab import files
files.download("/content/ds_demand.zip")